# 📊 Finance Performance & Budget Analysis
**Portfolio Project — Data Analyst**  
*12-month illustrative P&L dataset | Amounts in ₹ Lakhs*

---
**Analyst:** [Your Name]  
**Period:** April – March (FY)  
**Data Source:** `data/pnl_data.xlsx` (synthetic illustrative dataset)

---
## Objective
Evaluate revenue, profitability and operating-cost trends over 12 months and benchmark actual performance against an annual budget.  
Key business questions:
1. How did revenue and margins trend month-on-month?
2. Did actual results meet the annual budget — and why (or why not)?
3. What is the real management story hidden in the numbers?


## 1 · Setup & Data Load

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path

# ── Style ──────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi': 140,
    'figure.facecolor': '#FAFAFA',
    'axes.facecolor': '#FAFAFA',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.35,
    'grid.linestyle': '--',
    'font.family': 'sans-serif',
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.titleweight': 'bold',
})

DARK_BLUE  = '#17365D'
TEAL       = '#2F7E79'
AMBER      = '#C55A11'
PURPLE     = '#7030A0'
RED        = '#C00000'
GREEN_OK   = '#276221'
PALETTE    = [DARK_BLUE, TEAL, AMBER, PURPLE]

DATA_PATH = Path('../data/pnl_data.xlsx')
OUT_CHARTS = Path('../outputs/charts')
OUT_CHARTS.mkdir(parents=True, exist_ok=True)
print('Libraries loaded ✅')


## 2 · Load & Validate Data

In [ ]:
# ── Load Raw Data ──────────────────────────────────────────────────────────
raw = pd.read_excel(DATA_PATH, sheet_name='Raw Data', nrows=13)
raw = raw.dropna(subset=['Month'])
raw = raw[raw['Month'] != 'Note:']
raw = raw.reset_index(drop=True)

# Rename for clarity
raw.columns = raw.columns.str.strip()
print('Columns:', list(raw.columns))
print(f'\nRows: {len(raw)}  |  Months: {list(raw["Month"])}')
raw.head()


In [ ]:
# ── Load Budget Variance ───────────────────────────────────────────────────
bv = pd.read_excel(DATA_PATH, sheet_name='Budget Variance', nrows=8)
bv.columns = bv.columns.str.strip()
bv = bv.dropna(subset=['Metric'])
bv = bv[~bv['Metric'].isin(['Note:','Metric'])]
print(bv.to_string(index=False))


In [ ]:
# ── Null & Type Check ──────────────────────────────────────────────────────
print('Null counts:')
print(raw.isnull().sum())
print()
print('Data types:')
print(raw.dtypes)


## 3 · Feature Engineering
Build all calculated metrics from the raw inputs — same logic as the Excel P&L sheet,  
but now reproducible in Python.

> **Tax assumption:** A single corporate tax rate of **20%** is applied to Profit Before Tax (PBT).  
> Change `TAX_RATE` below to recalculate everything automatically.


In [ ]:
# ── Tax Assumption (single input cell) ────────────────────────────────────
TAX_RATE = 0.20   # ← change here to recalculate all downstream figures

df = raw.copy()

# ── P&L build ──────────────────────────────────────────────────────────────
df['Gross_Profit']   = df['Revenue'] - df['COGS']
df['Gross_Margin']   = df['Gross_Profit'] / df['Revenue']

df['Total_OpEx']     = df[['Salaries','Marketing','Rent','Utilities','Other Opex']].sum(axis=1)
df['OpEx_pct_Rev']   = df['Total_OpEx'] / df['Revenue']

df['EBITDA']         = df['Gross_Profit'] - df['Total_OpEx']
df['EBITDA_Margin']  = df['EBITDA'] / df['Revenue']

df['EBIT']           = df['EBITDA'] - df['Depreciation']
df['PBT']            = df['EBIT'] - df['Interest']
df['Tax_Calc']       = (df['PBT'] * TAX_RATE).round(2)
df['Net_Profit']     = df['PBT'] - df['Tax_Calc']
df['NP_Margin']      = df['Net_Profit'] / df['Revenue']

# ── Month-on-Month growth ──────────────────────────────────────────────────
df['Rev_MoM_Growth'] = df['Revenue'].pct_change()

# ── Month order ────────────────────────────────────────────────────────────
MONTH_ORDER = ['April','May','June','July','August','September',
               'October','November','December','January','February','March']
df['Month'] = pd.Categorical(df['Month'], categories=MONTH_ORDER, ordered=True)
df = df.sort_values('Month').reset_index(drop=True)

print('P&L built ✅  Shape:', df.shape)
df[['Month','Revenue','Gross_Profit','Gross_Margin','EBITDA','EBITDA_Margin',
    'Net_Profit','NP_Margin','Tax_Calc','OpEx_pct_Rev']].round(3)


## 4 · Full-Year KPI Summary

In [ ]:
# ── Compute full-year KPIs ─────────────────────────────────────────────────
fy_revenue     = df['Revenue'].sum()
fy_gp          = df['Gross_Profit'].sum()
fy_gm          = fy_gp / fy_revenue
fy_ebitda      = df['EBITDA'].sum()
fy_ebitda_m    = fy_ebitda / fy_revenue
fy_np          = df['Net_Profit'].sum()
fy_np_m        = fy_np / fy_revenue
fy_opex        = df['Total_OpEx'].sum()
fy_opex_pct    = fy_opex / fy_revenue

print('='*55)
print(f'  Full-Year KPIs (₹ Lakhs)          FY Actual')
print('='*55)
print(f'  Revenue                          ₹{fy_revenue:,.1f}L')
print(f'  Gross Profit                     ₹{fy_gp:,.1f}L')
print(f'  Gross Margin %                   {fy_gm:.1%}')
print(f'  EBITDA                           ₹{fy_ebitda:,.1f}L')
print(f'  EBITDA Margin %                  {fy_ebitda_m:.1%}')
print(f'  Net Profit                       ₹{fy_np:,.1f}L')
print(f'  Net Profit Margin %              {fy_np_m:.1%}')
print(f'  Total OpEx                       ₹{fy_opex:,.1f}L')
print(f'  OpEx as % of Revenue             {fy_opex_pct:.1%}')
print('='*55)


In [ ]:
# ── KPI Card Chart ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
fig.patch.set_facecolor('#17365D')

kpis = [
    ('Total Revenue',    f'₹{fy_revenue:,.0f}L', 'Full Year',    TEAL),
    ('Gross Margin',     f'{fy_gm:.1%}',          'Full Year',    AMBER),
    ('EBITDA',           f'₹{fy_ebitda:,.1f}L',   'Full Year',   PURPLE),
    ('Net Profit',       f'₹{fy_np:,.1f}L',        'Full Year',   '#E8A020'),
]
for ax, (title, value, sub, color) in zip(axes, kpis):
    ax.set_facecolor('#1E4080')
    ax.set_xlim(0,1); ax.set_ylim(0,1)
    ax.axis('off')
    ax.text(0.5, 0.72, title, ha='center', va='center', color='#A9C4E8',
            fontsize=11, fontweight='bold', transform=ax.transAxes)
    ax.text(0.5, 0.42, value, ha='center', va='center', color='white',
            fontsize=24, fontweight='bold', transform=ax.transAxes)
    ax.text(0.5, 0.16, sub, ha='center', va='center', color=color,
            fontsize=10, transform=ax.transAxes)
    for spine in ax.spines.values():
        spine.set_edgecolor(color)
        spine.set_linewidth(2)
        spine.set_visible(True)

fig.suptitle('FY Financial Performance Dashboard — ₹ Lakhs',
             color='white', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout(pad=0.8)
plt.savefig(OUT_CHARTS / 'kpi_cards.png', bbox_inches='tight',
            facecolor='#17365D', dpi=150)
plt.show()
print('Chart saved: outputs/charts/kpi_cards.png')


## 5 · Monthly Revenue Trend & Month-on-Month Growth

> **Key finding:** The often-cited "43.9% revenue growth" compares only April (₹82L) to March (₹118L).  
> A more defensible metric is the **compound MoM growth rate ≈ 3.4%**, computed below.


In [ ]:
avg_mom = df['Rev_MoM_Growth'].iloc[1:].mean()
cagr_mom = (df['Revenue'].iloc[-1] / df['Revenue'].iloc[0]) ** (1/11) - 1
endpoint_growth = (df['Revenue'].iloc[-1] - df['Revenue'].iloc[0]) / df['Revenue'].iloc[0]

print(f'Endpoint growth (March vs April)  : {endpoint_growth:.1%}')
print(f'Avg arithmetic MoM growth         : {avg_mom:.1%}')
print(f'Compound MoM growth (geometric)   : {cagr_mom:.1%}  ← most defensible')


In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 9), sharex=True,
                                 gridspec_kw={'height_ratios':[3,1.5]})

months_short = ['Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec','Jan','Feb','Mar']

# Bar chart — Revenue
bars = ax1.bar(months_short, df['Revenue'], color=DARK_BLUE, alpha=0.85,
               width=0.6, zorder=3, label='Revenue (₹L)')
ax1.plot(months_short, df['Revenue'], color=TEAL, marker='o', lw=2.5,
         markersize=6, zorder=4)
for bar in bars:
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.8,
             f'₹{bar.get_height():.0f}', ha='center', va='bottom',
             fontsize=9, color=DARK_BLUE, fontweight='bold')
ax1.axhline(df['Revenue'].mean(), color=AMBER, ls='--', lw=1.5,
            label=f'Monthly Avg ₹{df["Revenue"].mean():.0f}L')
ax1.set_ylabel('Revenue (₹ Lakhs)')
ax1.set_title('Monthly Revenue Trend')
ax1.legend(loc='upper left')
ax1.set_ylim(60, 135)

# MoM Growth bars
colors_mom = [GREEN_OK if x >= 0 else RED for x in df['Rev_MoM_Growth'].fillna(0)]
ax2.bar(months_short, df['Rev_MoM_Growth'].fillna(0) * 100,
        color=colors_mom, alpha=0.8, width=0.6, zorder=3)
ax2.axhline(0, color='black', lw=0.8)
ax2.axhline(cagr_mom * 100, color=AMBER, ls='--', lw=1.5,
            label=f'Compound MoM avg {cagr_mom:.1%}')
ax2.set_ylabel('MoM Growth (%)')
ax2.set_title('Month-on-Month Revenue Growth')
ax2.set_xlabel('Month')
ax2.legend(loc='lower right')
ax2.yaxis.set_major_formatter(mtick.FormatStrFormatter('%.0f%%'))

plt.tight_layout(pad=1.5)
plt.savefig(OUT_CHARTS / 'revenue_trend.png', bbox_inches='tight', dpi=150)
plt.show()


## 6 · Margin Trends (Gross / EBITDA / Net Profit)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(months_short, df['Gross_Margin']*100,   marker='o', color=DARK_BLUE,
        lw=2.5, markersize=7, label='Gross Margin %')
ax.plot(months_short, df['EBITDA_Margin']*100,  marker='s', color=TEAL,
        lw=2.5, markersize=7, label='EBITDA Margin %')
ax.plot(months_short, df['NP_Margin']*100,      marker='^', color=AMBER,
        lw=2.5, markersize=7, label='Net Profit Margin %')
ax.plot(months_short, df['OpEx_pct_Rev']*100,   marker='D', color=RED,
        lw=2, markersize=6, ls='--', label='OpEx as % of Revenue ⚠')

# Annotations at endpoints
for series, color, fmt in [
    ('Gross_Margin',  DARK_BLUE, '.1%'),
    ('EBITDA_Margin', TEAL,      '.1%'),
    ('NP_Margin',     AMBER,     '.1%'),
    ('OpEx_pct_Rev',  RED,       '.1%'),
]:
    val = df[series].iloc[-1]
    ax.annotate(f'{val:{fmt}}', xy=(11, val*100),
                xytext=(11.15, val*100), color=color,
                fontsize=9, fontweight='bold', va='center')

ax.set_ylabel('Margin (%)')
ax.set_xlabel('Month')
ax.set_title('Monthly Margin Trends — Gross / EBITDA / Net Profit vs OpEx %')
ax.legend(loc='upper left', fontsize=10)
ax.yaxis.set_major_formatter(mtick.FormatStrFormatter('%.0f%%'))

plt.tight_layout()
plt.savefig(OUT_CHARTS / 'margin_trends.png', bbox_inches='tight', dpi=150)
plt.show()


## 7 · Budget vs Actual Variance Analysis
Colour-coded: 🟢 **Favourable** | 🔴 **Unfavourable**  
*(A positive variance is NOT always favourable — higher COGS/OpEx is unfavourable despite being a positive number)*


In [ ]:
# Define budget targets
budget = {
    'Revenue':          1150,
    'COGS':             630,
    'Gross Profit':     520,
    'Operating Expenses': 275,
    'EBITDA':           245,
    'Net Profit':       145,
}
actual = {
    'Revenue':           df['Revenue'].sum(),
    'COGS':              df['COGS'].sum(),
    'Gross Profit':      df['Gross_Profit'].sum(),
    'Operating Expenses':df['Total_OpEx'].sum(),
    'EBITDA':            df['EBITDA'].sum(),
    'Net Profit':        df['Net_Profit'].sum(),
}

# Favourable if positive variance for: Revenue, GP, EBITDA, NP
# Unfavourable if positive variance for: COGS, OpEx
fav_if_positive = {
    'Revenue': True, 'COGS': False, 'Gross Profit': True,
    'Operating Expenses': False, 'EBITDA': True, 'Net Profit': True,
}

rows = []
for metric in budget:
    b = budget[metric]
    a = actual[metric]
    var = a - b
    var_pct = var / b
    is_fav = (var > 0 and fav_if_positive[metric]) or              (var < 0 and not fav_if_positive[metric])
    rows.append({
        'Metric': metric,
        'Budget (₹L)': b,
        'Actual (₹L)': round(a, 1),
        'Variance (₹L)': round(var, 1),
        'Variance %': f'{var_pct:+.1%}',
        'Signal': '✅ FAV' if is_fav else '❌ UNFAV',
    })

bv_df = pd.DataFrame(rows)

def colour_row(row):
    is_fav = row['Signal'] == '✅ FAV'
    colour = 'background-color: #C6EFCE; color: #276221' if is_fav              else 'background-color: #FFC7CE; color: #9C0006'
    return [colour if col in ['Variance (₹L)','Variance %','Signal'] else ''
            for col in row.index]

bv_df.style.apply(colour_row, axis=1).format({'Budget (₹L)': '{:.1f}', 'Actual (₹L)': '{:.1f}'})


In [ ]:
# ── Grouped Bar Chart: Budget vs Actual ───────────────────────────────────
metrics = list(budget.keys())
b_vals = [budget[m] for m in metrics]
a_vals = [round(actual[m], 1) for m in metrics]
x = np.arange(len(metrics))
width = 0.36

fig, ax = plt.subplots(figsize=(14, 6))
bars_b = ax.bar(x - width/2, b_vals, width, label='Budget', color=DARK_BLUE, alpha=0.85)
bars_a = ax.bar(x + width/2, a_vals, width, label='Actual',
                color=[('#276221' if (actual[m]-budget[m]>0 and fav_if_positive[m]) or
                                    (actual[m]-budget[m]<0 and not fav_if_positive[m])
                        else '#C00000') for m in metrics],
                alpha=0.85)

# Variance labels on top
for i, m in enumerate(metrics):
    var = actual[m] - budget[m]
    is_fav = (var > 0 and fav_if_positive[m]) or (var < 0 and not fav_if_positive[m])
    colour = '#276221' if is_fav else '#C00000'
    label  = f'+{var:.1f}' if var >= 0 else f'{var:.1f}'
    ax.text(x[i] + width/2, max(b_vals[i], a_vals[i]) + 5, label,
            ha='center', fontsize=9, color=colour, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(metrics, rotation=15, ha='right')
ax.set_ylabel('₹ Lakhs')
ax.set_title('Budget vs Actual — Full Year (₹ Lakhs)')
ax.legend()
ax.set_ylim(0, max(max(b_vals), max(a_vals)) * 1.18)

plt.tight_layout()
plt.savefig(OUT_CHARTS / 'budget_vs_actual.png', bbox_inches='tight', dpi=150)
plt.show()


## 8 · Operating Expense Deep-Dive
**Headline finding:** OpEx exceeded budget by **+9.6%** while revenue only grew **+3.4%** above budget.  
This is the primary reason EBITDA missed target despite a revenue beat.


In [ ]:
# ── OpEx composition (stacked bar) ───────────────────────────────────────
opex_cols = ['Salaries','Marketing','Rent','Utilities','Other Opex']
opex_palette = [DARK_BLUE, TEAL, AMBER, PURPLE, '#C55A11']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Stacked bar
bottom = np.zeros(12)
for col, color in zip(opex_cols, opex_palette):
    ax1.bar(months_short, df[col], bottom=bottom, label=col, color=color, alpha=0.85)
    bottom += df[col].values

ax1.plot(months_short, df['Total_OpEx'], color='black', marker='o',
         lw=2, markersize=5, label='Total OpEx', zorder=5)
ax1.set_title('OpEx Composition by Month (Stacked)')
ax1.set_ylabel('₹ Lakhs')
ax1.legend(fontsize=9, loc='upper left')
ax1.set_ylim(0, df['Total_OpEx'].max() * 1.3)

# OpEx % Revenue line + budget benchmark
budget_opex_pct = 275 / 1150
ax2.plot(months_short, df['OpEx_pct_Rev']*100, marker='o', color=RED,
         lw=2.5, markersize=7, label='Actual OpEx % Revenue')
ax2.axhline(budget_opex_pct*100, color=DARK_BLUE, ls='--', lw=2,
            label=f'Budget benchmark {budget_opex_pct:.1%}')
ax2.fill_between(months_short, df['OpEx_pct_Rev']*100, budget_opex_pct*100,
                 where=(df['OpEx_pct_Rev'] > budget_opex_pct),
                 alpha=0.15, color=RED, label='Over budget zone')
ax2.set_title('OpEx as % of Revenue vs Budget Benchmark')
ax2.set_ylabel('OpEx / Revenue (%)')
ax2.yaxis.set_major_formatter(mtick.FormatStrFormatter('%.0f%%'))
ax2.legend(fontsize=9)

plt.tight_layout()
plt.savefig(OUT_CHARTS / 'opex_analysis.png', bbox_inches='tight', dpi=150)
plt.show()


## 9 · Tax Modelling — Assumption-Driven
Instead of hardcoded monthly tax values (which caused the effective rate to drift from 23% → 15%),  
tax is now calculated from a **single `TAX_RATE` input** set in Section 3.

Below we show the effective tax rate at different assumption levels.


In [ ]:
print(f'Current TAX_RATE assumption: {TAX_RATE:.0%}')
print()
print(f'{'Month':<12} {'PBT (₹L)':>10} {'Tax @ {:.0%} (₹L)'.format(TAX_RATE):>16} {'Eff. Rate':>10}')
print('-'*52)
for _, row in df.iterrows():
    eff = row['Tax_Calc'] / row['PBT'] if row['PBT'] else 0
    print(f'{row["Month"]:<12} {row["PBT"]:>10.2f} {row["Tax_Calc"]:>16.2f} {eff:>10.1%}')
print('-'*52)
print(f'{'Total':<12} {df["PBT"].sum():>10.2f} {df["Tax_Calc"].sum():>16.2f} '
      f'{df["Tax_Calc"].sum()/df["PBT"].sum():>10.1%}')


In [ ]:
# Sensitivity: Net Profit at different tax rates
rates   = [0.15, 0.18, 0.20, 0.22, 0.25, 0.30]
np_vals = []
for r in rates:
    tax = (df['PBT'] * r).round(2)
    np_vals.append((df['PBT'] - tax).sum())

fig, ax = plt.subplots(figsize=(10, 5))
bar_colors = ['#C6EFCE' if v >= df['Net_Profit'].sum() else '#FFC7CE' for v in np_vals]
ax.bar([f'{r:.0%}' for r in rates], np_vals, color=bar_colors, alpha=0.9, edgecolor='white')
ax.axhline(df['Net_Profit'].sum(), color=DARK_BLUE, ls='--', lw=1.5,
           label=f'Current NP @ {TAX_RATE:.0%}: ₹{df["Net_Profit"].sum():.1f}L')
for i, (r, v) in enumerate(zip(rates, np_vals)):
    ax.text(i, v + 0.5, f'₹{v:.1f}L', ha='center', fontsize=9, fontweight='bold')
ax.set_xlabel('Corporate Tax Rate Assumption')
ax.set_ylabel('Full-Year Net Profit (₹ Lakhs)')
ax.set_title('Tax Rate Sensitivity — Impact on Net Profit')
ax.legend()
plt.tight_layout()
plt.savefig(OUT_CHARTS / 'tax_sensitivity.png', bbox_inches='tight', dpi=150)
plt.show()


## 10 · Management Insights & Summary Statistics

In [ ]:
from scipy import stats as sp_stats

print('='*65)
print('  MANAGEMENT INSIGHTS SUMMARY')
print('='*65)

# Revenue
print('\n📈  REVENUE')
print(f'  Peak month   : {df.loc[df["Revenue"].idxmax(),"Month"]} (₹{df["Revenue"].max():.0f}L)')
print(f'  Lowest month : {df.loc[df["Revenue"].idxmin(),"Month"]} (₹{df["Revenue"].min():.0f}L)')
print(f'  Endpoint growth (Apr→Mar) : {endpoint_growth:.1%}')
print(f'  Avg MoM growth (arithmetic): {avg_mom:.1%}')
print(f'  Compound MoM growth        : {cagr_mom:.1%}  ← preferred metric')

# Margins
print('\n📊  MARGINS (Full Year)')
print(f'  Gross Margin   : {fy_gm:.1%}')
print(f'  EBITDA Margin  : {fy_ebitda_m:.1%}')
print(f'  Net Profit Margin: {fy_np_m:.1%}')
print(f'  OpEx % Revenue : {fy_opex_pct:.1%}  (budget: {budget_opex_pct:.1%})')

# Budget
print('\n🎯  BUDGET PERFORMANCE')
rev_var  = actual["Revenue"] - budget["Revenue"]
opex_var = actual["Operating Expenses"] - budget["Operating Expenses"]
ebt_var  = actual["EBITDA"] - budget["EBITDA"]
np_var   = actual["Net Profit"] - budget["Net Profit"]
print(f'  Revenue  : {rev_var:+.1f}L ({rev_var/budget["Revenue"]:+.1%}) ✅ FAV')
print(f'  OpEx     : {opex_var:+.1f}L ({opex_var/budget["Operating Expenses"]:+.1%}) ❌ UNFAV  ← KEY RISK')
print(f'  EBITDA   : {ebt_var:+.1f}L ({ebt_var/budget["EBITDA"]:+.1%}) ❌ UNFAV')
print(f'  Net Profit: {np_var:+.1f}L ({np_var/budget["Net Profit"]:+.1%}) ✅ FAV (windfall)')

# Below-EBITDA explanation
actual_below_ebitda = df['EBITDA'].sum() - df['Net_Profit'].sum()
budget_below_ebitda = budget['EBITDA'] - budget['Net Profit']
windfall = budget_below_ebitda - actual_below_ebitda
print('\n⚡  BELOW-EBITDA GAP EXPLAINED')
print(f'  Implied budget below-EBITDA costs: ₹{budget_below_ebitda:.1f}L')
print(f'  Actual below-EBITDA costs        : ₹{actual_below_ebitda:.1f}L')
print(f'  Cost windfall below EBITDA        : ₹{windfall:.1f}L')
print(f'  → This ₹{windfall:.1f}L saving converted an EBITDA miss into a net profit beat.')
print('='*65)


## 11 · Correlation Analysis

In [ ]:
corr_cols = ['Revenue','Gross_Profit','Total_OpEx','EBITDA','Net_Profit',
             'Gross_Margin','EBITDA_Margin','NP_Margin','OpEx_pct_Rev']
corr = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn', vmin=-1, vmax=1,
            square=True, linewidths=0.5, ax=ax,
            annot_kws={'size': 9})
ax.set_title('Correlation Matrix — Key Financial Metrics')
plt.tight_layout()
plt.savefig(OUT_CHARTS / 'correlation_heatmap.png', bbox_inches='tight', dpi=150)
plt.show()


## 12 · Export Summary Report to Excel

In [ ]:
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils.dataframe import dataframe_to_rows

REPORT_PATH = Path('../outputs/reports/PnL_Summary_Report.xlsx')

wb_out = Workbook()
ws_fy  = wb_out.active
ws_fy.title = 'Full-Year Summary'

header_fill = PatternFill('solid', fgColor='17365D')
header_font = Font(color='FFFFFF', bold=True)
fav_fill    = PatternFill('solid', fgColor='C6EFCE')
unfav_fill  = PatternFill('solid', fgColor='FFC7CE')

# ── Sheet 1: Full-Year KPIs ────────────────────────────────────────────────
ws_fy.append(['Metric', 'Value', 'Unit'])
for r in ws_fy.iter_rows(min_row=1, max_row=1):
    for c in r:
        c.font = header_font; c.fill = header_fill

kpi_rows = [
    ('Revenue',            fy_revenue,    '₹ Lakhs'),
    ('Gross Profit',       fy_gp,         '₹ Lakhs'),
    ('Gross Margin',       f'{fy_gm:.1%}','%'),
    ('EBITDA',             fy_ebitda,     '₹ Lakhs'),
    ('EBITDA Margin',      f'{fy_ebitda_m:.1%}','%'),
    ('Net Profit',         fy_np,         '₹ Lakhs'),
    ('Net Profit Margin',  f'{fy_np_m:.1%}','%'),
    ('OpEx % Revenue',     f'{fy_opex_pct:.1%}','%'),
    ('Tax Rate Assumed',   f'{TAX_RATE:.0%}','%'),
]
for row in kpi_rows:
    ws_fy.append(list(row))
ws_fy.column_dimensions['A'].width = 22
ws_fy.column_dimensions['B'].width = 14

# ── Sheet 2: Monthly Data ──────────────────────────────────────────────────
ws_mo = wb_out.create_sheet('Monthly Data')
export_cols = ['Month','Revenue','Gross_Profit','Gross_Margin','Total_OpEx',
               'OpEx_pct_Rev','EBITDA','EBITDA_Margin','Net_Profit','NP_Margin',
               'PBT','Tax_Calc']
export_df = df[export_cols].copy()
export_df.columns = ['Month','Revenue','Gross Profit','Gross Margin %','Total OpEx',
                      'OpEx % Rev','EBITDA','EBITDA Margin %','Net Profit',
                      'NP Margin %','PBT','Tax (calc)']

for r_idx, row in enumerate(dataframe_to_rows(export_df, index=False, header=True), 1):
    ws_mo.append(row)
    if r_idx == 1:
        for c in ws_mo[1]:
            c.font = header_font; c.fill = header_fill

# ── Sheet 3: Budget vs Actual ──────────────────────────────────────────────
ws_bv_out = wb_out.create_sheet('Budget vs Actual')
for r_idx, row in enumerate(dataframe_to_rows(bv_df, index=False, header=True), 1):
    ws_bv_out.append(row)
    if r_idx == 1:
        for c in ws_bv_out[1]:
            c.font = header_font; c.fill = header_fill
    else:
        signal = bv_df.iloc[r_idx-2]['Signal']
        fill = fav_fill if 'FAV' in signal else unfav_fill
        for c in ws_bv_out[r_idx]:
            if c.column in [4,5,6]:
                c.fill = fill

wb_out.save(REPORT_PATH)
print(f'Report saved: {REPORT_PATH}')


## 13 · Conclusions & Recommendations

### What the numbers say
| Finding | Detail |
|---|---|
| ✅ Revenue beat | +₹39L (+3.4%) above budget — steady growth throughout the year |
| ❌ OpEx overrun | +₹26.5L (+9.6%) above budget — grew **3× faster than revenue** |
| ❌ EBITDA miss | -₹6.5L (-2.6%) — entirely caused by OpEx overrun |
| ✅ Net profit beat | +₹25.1L (+17.3%) — driven by below-EBITDA cost windfall, **not** trading outperformance |
| ⚠ Tax drift fixed | Hardcoded tax created a drifting effective rate (23%→15%). Now modelled at flat 20% |
| 📈 Growth metric | Endpoint 43.9% is misleading. Compound MoM rate ≈ **3.4%** is more defensible |

### Recommendations
1. **Cost control is the #1 priority.** OpEx % Revenue ran at 25.4% vs 23.9% budget. This gap must close for sustainable EBITDA growth.
2. **Do not rely on below-EBITDA windfalls.** The ₹31.6L saving in depreciation, interest and tax flattered net profit. This may not repeat.
3. **Align tax reporting.** Use a single rate assumption and recalculate; avoid month-by-month manual entries.
4. **Watch November and December.** Revenue dipped month-on-month (-6.3% and -3.8%), suggesting seasonality — plan OpEx accordingly.

---
*This notebook is a portfolio project demonstrating: data extraction from Excel, pandas P&L modelling, matplotlib/seaborn visualisation, sensitivity analysis, and structured financial storytelling.*
